# 01 — Data Cleaning


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/online_retail_II.csv", encoding='ISO-8859-1')

In [3]:
df.shape

(541910, 8)

In [4]:
df.describe()

,Quantity,Price,Customer ID
count,541910.000000,541910.000000,406830.000000
mean,9.552234,4.611138,15287.684160
std,218.080957,96.759765,1713.603074
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  str    
 1   StockCode    541910 non-null  str    
 2   Description  540456 non-null  str    
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  str    
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [6]:
df.sample(15)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
970,536520,22754,SMALL RED BABUSHKA NOTEBOOK,1,12/1/10 12:43,0.85,14729.0,United Kingdom
354604,567887,22144,CHRISTMAS CRAFT LITTLE FRIENDS,6,9/22/11 15:28,2.10,15237.0,United Kingdom
217581,555925,23050,RECYCLED ACAPULCO MAT GREEN,2,6/7/11 17:13,16.63,NaN,United Kingdom
434959,574066,85150,LADIES & GENTLEMEN METAL SIGN,18,11/2/11 14:50,2.55,13183.0,United Kingdom
346615,567192,23445,ICE CREAM BUBBLES,1,9/19/11 9:20,0.83,17961.0,United Kingdom
121851,546789,20711,JUMBO BAG TOYS,600,3/17/11 10:17,1.65,15769.0,United Kingdom
132652,547684,22746,POPPY'S PLAYHOUSE LIVINGROOM,6,3/24/11 14:46,2.10,12408.0,Belgium
340881,566723,22466,FAIRY TALE COTTAGE NIGHT LIGHT,12,9/14/11 13:02,1.95,15804.0,United Kingdom
324241,565396,22703,PINK CAT BOWL,3,9/2/11 16:39,1.63,NaN,United Kingdom
257864,559547,23284,DOORMAT KEEP CALM AND COME IN,3,7/10/11 15:12,7.95,17758.0,United Kingdom


→ There are some stockcodes like C543974 which have c prefix and their (quantity<0 and price>0) which are cancled invoices that should not be included in total revenue.

In [7]:
df[df['Invoice'].astype(str).str.contains('[A-Za-z]', regex=True, na=False)].value_counts(['Invoice'])


Invoice
C570867    101
C560540     57
C548460     45
C560855     41
C538341     39
          ... 
C581463      1
C581470      1
C581484      1
C581499      1
C581568      1
Name: count, Length: 3839, dtype: int64

check what types of prefixs we do have for invoices

In [8]:
df[~df['Invoice'].str.isnumeric()]['Invoice'].str[0].value_counts()

Invoice
C    9288
A       3
Name: count, dtype: int64

→ There are some stockcodes like A563185 which have a prefix mean that the bad debt has been adjusted, they are not included in total revenue

In [9]:
df[df['Invoice'].astype(str).str.startswith('A', na=False)].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
299982,A563185,B,Adjust bad debt,1,8/12/11 14:50,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,8/12/11 14:51,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,8/12/11 14:52,-11062.06,NaN,United Kingdom


In [10]:
df[df['Invoice'].str.startswith('C', na=False)]['Quantity'].describe()

count     9288.000000
mean       -29.885228
std       1145.786965
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

→ C-invoices max is -1 which we gives us the hint that all cancelations are negative which is logical.

In [11]:
unflagged_cancellations = df[(df['Quantity'] < 0) & (df['Price'] < 0) & (~df['Invoice'].astype(str).str.startswith(('C', 'A'), na=False))]
unflagged_cancellations

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country


→ We want to check if there are any invoices numbers that are cancled invoices bt have not been flagged \
be ebarati they are canceled bt have not been flagged canceled



In [12]:
unflagged_negative_qty = df[(df['Quantity'] < 0) & (df['Price'] == 0)]
unflagged_negative_qty

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
2406,536589,21777,NaN,-10,12/1/10 16:50,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,12/2/10 14:42,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,12/3/10 15:30,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,12/7/11 18:36,0.0,NaN,United Kingdom
535335,581212,22578,lost,-1050,12/7/11 18:38,0.0,NaN,United Kingdom
535336,581213,22576,check,-30,12/7/11 18:38,0.0,NaN,United Kingdom
536910,581226,23090,missing,-338,12/8/11 9:56,0.0,NaN,United Kingdom


In [13]:
zero_price_positive_qty = df[(df['Price'] == 0) & (df['Quantity'] > 0)]
print(f"Rows with Price=0, Quantity>0: {len(zero_price_positive_qty)}")
print(f"  - no Customer ID :   {zero_price_positive_qty['Customer ID'].isna().sum()}")
print(f"  - has Customer ID :  {zero_price_positive_qty['Customer ID'].notna().sum()}")

Rows with Price=0, Quantity>0: 1179
  - no Customer ID :   1139
  - has Customer ID :  40


there are some gifts given to customers --> 40 invoices\
also there are some invoices that were sold items with zero price with no cutId --> likely stock adjustments? not sure

In [14]:
promotional_invoices = df[(df['Quantity'] > 0) & (df['Price'] == 0)]
promotional_invoices.sample(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
314747,564651,22955,36 FOIL STAR CAKE CASES,144,8/26/11 14:19,0.0,14646.0,Netherlands
41494,539856,22367,CHILDRENS APRON SPACEBOY DESIGN,2,12/22/10 14:41,0.0,NaN,United Kingdom
178279,552230,84375,SET OF 20 KIDS COOKIE CUTTERS,1,5/6/11 15:43,0.0,NaN,United Kingdom
418658,572739,21578,found,16,10/25/11 15:44,0.0,NaN,United Kingdom
14340,537534,22697,GREEN REGENCY TEACUP AND SAUCER,1,12/7/10 11:48,0.0,NaN,United Kingdom


40 invoices which items which were given for free

In [15]:
a_invoices = df[df['Invoice'].str.startswith('A', na=False)]
c_invoices = df[df['Invoice'].str.startswith('C', na=False)]

In [16]:
df.isna().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [17]:
df.duplicated().sum()

5268

In [18]:
df.nunique()

Invoice        25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
Price           1630
Customer ID     4372
Country           38
dtype: int64

## Summary of issues found

1 - invoice date is str it should become datetime\
2 - customer id is float while we don't need floating points\
3 - customerId is null for 135,080 rows — kept for revenue analysis bt dropped for customer analysis\
4 - There are negative values for quantity + price --> all the negative values are cancelations \
5 - Description is null for 1,454 rows\
6 - Quantity has negative values (min -80995),returns/cancellations or stock adjustments\
7 - There are 5,268 duplicate rows \
8 - There are 1,179 rows with Price = 0 and Quantity > 0 (not caught by the existing negative-quantity write-off filter) — 1,139 are internal notes with no Customer ID, 40 are real customer orders receiving a free/promotional item

## Cleaning steps applied
1. Dropped exact duplicates
2. Dropped 'A' (bad-debt) invoices
3. Dropped negative-qty zero-price write-offs (lost/broken/check) and no-cutId zero-price rows
4. Converted InvoiceDate → datetime, Customer ID → int
5. Flagged cancellations (`is_cancelled`)
6. Filled missing Description with 'Unknown'
7. Added `total_payment` (Quantity × Price)
8. Flagged the remaining 40 Price=0/Quantity>0 rows with `is_zero_value` (real free items on real orders, kept but excluded from order/frequency counts downstream)

In [19]:
df = df.drop_duplicates()

5268 rows dropped

In [20]:
df = df.drop(a_invoices.index)

removed the invocies which were bad depth

### Zero-value rows (Price = 0, Quantity > 0)
no cutId --> internal note, drop. has cutId --> real free item on a real order, keep + flag

In [21]:
df = df.drop(unflagged_negative_qty.index)

In [22]:
zero_no_cust = df[(df['Price'] == 0) & (df['Quantity'] > 0) & (df['Customer ID'].isna())]
df = df.drop(zero_no_cust.index)

dropped lost/broken/check write-offs and the no-cutId zero-price rows

In [23]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%y %H:%M') # keep full datetime (hour/minute) for hourly-level analysis

In [24]:
df['Customer ID'] = df['Customer ID'].astype('Int64')

→ Int64 (nullable) instead of int64 to keep the remaining nulls without erroring.

In [25]:
df['is_cancelled'] = df['Invoice'].str.startswith('C', na=False)

In [26]:
df['is_zero_value'] = (df['Price'] == 0) & (df['Quantity'] > 0)
df['is_zero_value'].sum()

40

In [27]:
df['Description'] = df['Description'].fillna('Unknown')

In [28]:
df.isna().sum()


Invoice               0
StockCode             0
Description           0
Quantity              0
InvoiceDate           0
Price                 0
Customer ID      132564
Country               0
is_cancelled          0
is_zero_value         0
dtype: int64

In [29]:
df.duplicated().sum()

0

No missing Description, no duplicates. Remaining Customer ID nulls are by design (guest orders, kept for revenue-level analysis).

### total payment
`Quantity × Price` per row — computed once here so it's already in both processed CSVs, instead of recreating it in every downstream notebook.

In [30]:
df['total_payment'] = df['Quantity'] * df['Price']

In [31]:
revenue_df = df.copy()
customer_df = df.dropna(subset=['Customer ID'])

revenue_df.to_csv('../data/processed/revenue_df.csv', index=False)
customer_df.to_csv('../data/processed/customer_df.csv', index=False)

## Takeaways

Removed before analysis: 5,268 duplicate rows, 3 bad-debt 'A' invoices, 1,336 negative-qty zero-price write-offs, and ~1,134 no-cutId zero-price rows. None are real customer transactions.

40 zero-price rows have a real cutId --> free items on real orders. Kept but flagged `is_zero_value` so they don't count as purchases downstream.

~25% of rows have no cutId --> guest checkouts, real revenue we just can't tie to a customer. That's why we export two files: `revenue_df` (everything) and `customer_df` (cutId only).

Cancellations (~9,288 invoices) are flagged not dropped, so 02 can still compute Net Sales and return rate.